# P1b: read-only archive audit

Run after the frozen P1b analysis. This CPU notebook revalidates every sealed decision/loss artifact, the frozen router state, both completion reports, the selected-loss CSV, and the analysis report. It performs no model inference, no method revision, and no alternative scientific analysis.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == 'covsafe' or module_name.startswith('covsafe.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
importlib.import_module('covsafe')
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Git commit:', GIT_COMMIT)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path('/content/drive/MyDrive/covariate-safe-tsfm/private_manifests')
P1B_ROOT = PRIVATE_ROOT / 'p1b'
ANALYSIS_PATH = P1B_ROOT / 'reports' / 'p1b_sealed_analysis.json'
assert ANALYSIS_PATH.is_file(), f'Missing frozen analysis: {ANALYSIS_PATH}'
print('Frozen analysis found.')

In [ ]:
import json
from datetime import UTC, datetime

from covsafe.p1b import (
    EXPECTED_P1B_CONFIG_HASH,
    atomic_json,
    scientific_code_hash,
    sha256_file,
)
from covsafe.p1b_analysis import read_verified_sealed_artifacts

analysis = json.loads(ANALYSIS_PATH.read_text(encoding='utf-8'))
config, decisions, losses, state_sha256, inventory_sha256 = (
    read_verified_sealed_artifacts(REPO, P1B_ROOT, P1B_ROOT)
)
code_sha256 = scientific_code_hash(REPO)
selected_info = analysis['selected_loss_artifact']
selected_path = P1B_ROOT / selected_info['relative_path']
with selected_path.open('r', encoding='utf-8') as handle:
    selected_row_count = max(sum(1 for _ in handle) - 1, 0)

completion_reports = {}
for backbone in ('chronos_2', 'timesfm_3'):
    path = P1B_ROOT / 'reports' / f'{backbone}_p1b_completion.json'
    completion_reports[backbone] = json.loads(path.read_text(encoding='utf-8'))

checks = {
    'analysis_config_hash_matches': analysis['config_hash'] == EXPECTED_P1B_CONFIG_HASH,
    'analysis_scientific_code_hash_matches': analysis['scientific_code_sha256'] == code_sha256,
    'analysis_router_state_hash_matches': analysis['router_state_sha256'] == state_sha256,
    'analysis_sealed_inventory_hash_matches': (
        analysis['sealed_evaluation_inventory_sha256'] == inventory_sha256
    ),
    'decision_row_count_matches': analysis['counts']['decision_rows'] == len(decisions),
    'loss_row_count_matches': analysis['counts']['exhaustive_policy_loss_rows'] == len(losses),
    'selected_artifact_exists': selected_path.is_file(),
    'selected_artifact_hash_matches': selected_info['sha256'] == sha256_file(selected_path),
    'selected_artifact_row_count_matches': selected_info['row_count'] == selected_row_count,
    'first_fixed_sequence_test_failed': not analysis['fixed_sequence'][
        'step_1_gated_vs_fullset_best_constant'
    ]['passed'],
    'second_test_not_inferentially_entered': not analysis['fixed_sequence'][
        'step_2_gated_vs_ungated'
    ]['inferentially_tested'],
    'paper_success_gate_failed': not analysis['decision']['paper_success_gate_passed'],
}
for backbone, completion in completion_reports.items():
    checks[f'{backbone}_config_hash_matches'] = (
        completion['config_hash'] == EXPECTED_P1B_CONFIG_HASH
    )
    checks[f'{backbone}_scientific_code_hash_matches'] = (
        completion['scientific_code_sha256'] == code_sha256
    )
    checks[f'{backbone}_router_state_hash_matches'] = (
        completion['router_state_sha256'] == state_sha256
    )
    checks[f'{backbone}_completion_count_matches'] = (
        completion['completed_origin_count'] == 22
        and completion['evaluated_policy_run_count'] == 308
    )
assert all(checks.values()), {key: value for key, value in checks.items() if not value}
print('All immutable-artifact checks passed.')

In [ ]:
audit = {
    'audit': 'p1b-read-only-archive-audit',
    'schema_version': 1,
    'result_status': 'verified',
    'created_at_utc': datetime.now(UTC).isoformat(),
    'audit_git_commit': GIT_COMMIT,
    'config_hash': EXPECTED_P1B_CONFIG_HASH,
    'scientific_code_sha256': code_sha256,
    'analysis_report_sha256': sha256_file(ANALYSIS_PATH),
    'router_state_sha256': state_sha256,
    'sealed_evaluation_inventory_sha256': inventory_sha256,
    'selected_loss_artifact_sha256': selected_info['sha256'],
    'counts': analysis['counts'],
    'checks': checks,
    'scope': {
        'model_inference_performed': False,
        'scientific_analysis_recomputed': False,
        'method_or_threshold_revised': False,
        'source_artifacts_modified': False,
    },
}
AUDIT_PATH = P1B_ROOT / 'reports' / 'p1b_archive_audit.json'
atomic_json(AUDIT_PATH, audit)
print(json.dumps(audit, indent=2, ensure_ascii=False))

## Return artifact

Send the complete compact JSON above. This closes the integrity chain; it does not reopen method selection or authorize another analysis of the same sealed outcomes.